In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

In [2]:
def load_amira_lattice_float2(path, shape=(512, 512, 1001), dtype=np.dtype('<f4')):
    """
    Loads an AmiraMesh Lattice { float[2] Data } @1
    Returns: data shaped (nz, ny, nx, 2) by default (time as z).
    """
    nx, ny, nz = shape  # from "define Lattice 512 512 1001"

    with open(path, "rb") as f:
        raw = f.read()

    # Find the '@1' marker, then skip to the start of binary data after the newline
    marker = raw.find(b"@1")
    if marker == -1:
        raise ValueError("Could not find '@1' data marker in file.")

    # Data starts after '@1' and the following newline(s)
    data_start = marker + 2
    while data_start < len(raw) and raw[data_start] in (ord('\n'), ord('\r'), ord(' '), ord('\t')):
        data_start += 1

    # Interpret the remaining bytes as little-endian float32
    arr = np.frombuffer(raw, dtype=dtype, offset=data_start)

    expected = nx * ny * nz * 2
    if arr.size < expected:
        raise ValueError(
            f"File truncated. Got {arr.size}, expected at least {expected}."
        )

    arr = arr[:expected]  # Ignore trailing padding

    # Amira Lattice typically stores x fastest, then y, then z
    # So reshape as (nz, ny, nx, 2)
    data = arr.reshape((nz, ny, nx, 2))
    return data

In [3]:
# -------------------------
# Load data
# -------------------------
data = load_amira_lattice_float2("0001.am", shape=(512, 512, 1001))
print(f"Data shape: {data.shape}")  # (1001, 512, 512, 2)

# Extract velocity components
# data shape: (time, y, x, 2)
u = data[:, :, :, 0]  # x-component
v = data[:, :, :, 1]  # y-component
speed = np.sqrt(u**2 + v**2)

print(f"Speed shape: {speed.shape}")
print(f"Speed range: {speed.min():.4f} to {speed.max():.4f}")

Data shape: (1001, 512, 512, 2)


C:\Users\darsh\AppData\Local\Temp\ipykernel_23108\3052900352.py:11: RuntimeWarning: overflow encountered in square
  speed = np.sqrt(u**2 + v**2)


Speed shape: (1001, 512, 512)
Speed range: 0.0007 to inf


In [4]:
# -------------------------
# Create figure and animation
# -------------------------
vmin = float(speed.min())
vmax = float(speed.max())

fig, ax = plt.subplots(figsize=(10, 8))

# Initial frame - use mean across all timesteps for better representation
frame0 = speed.mean(axis=0)
quad = ax.imshow(frame0, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)

cbar = fig.colorbar(quad, ax=ax, label="Speed magnitude")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Turbulent flow speed (mean across time)")

# -------------------------
# Update function
# -------------------------
def update(frame_index):
    frame = speed[frame_index]
    quad.set_array(frame)
    ax.set_title(f"Turbulent flow speed at t-index {frame_index}")
    return quad,

# -------------------------
# Build animation
# -------------------------
anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(speed),
    interval=0.1,   # milliseconds per frame
    blit=False
)

# -------------------------
# Save as GIF
# -------------------------
anim.save("turbulent_flow_animation.gif", writer="pillow", fps=60)

plt.close(fig)
print("Animation saved as turbulent_flow_animation.gif")

Animation saved as turbulent_flow_animation.gif
